# Daily-Derived v1 正式 Stage 5 Hybrid 训练入口

本 Notebook 与 Raw Baseline 共用现有 Hybrid Stage 5 实现，但显式使用 Daily-Derived v1 的 16-leaf vocabulary、表达式 tensor、N=1/2 Exact-TB 资源和独立输出目录。所有真实训练默认关闭。

## 1. 环境与项目根目录

第一格负责 Notebook import bootstrap，并确认 CUDA 环境。

In [ ]:
import json
import platform
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import torch

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents) if (path / 'factor_gfn').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('无法从当前目录向上找到包含 factor_gfn/ 的项目根目录')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = 'cuda:0'
SEED = 42
CUDA_READY = torch.cuda.is_available()
ENVIRONMENT = {
    'project_root': str(PROJECT_ROOT),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': CUDA_READY,
    'gpu_name': torch.cuda.get_device_name(0) if CUDA_READY else None,
    'device': DEVICE,
    'seed': SEED,
}
print(json.dumps(ENVIRONMENT, ensure_ascii=False, indent=2), flush=True)
if not CUDA_READY:
    print('CUDA 不可用：可以检查 CPU-safe preflight，但正式训练仍被阻止。', flush=True)

## 2. 冻结训练合同

除 Feature Space 外，全部训练参数与 Raw Baseline 保持一致。

In [ ]:
from factor_gfn.gfn import (
    ExhaustiveRegistry,
    ExpressionFeatureSpec,
    HybridVarianceTrainer,
    RealRewardDataConfig,
    RealRewardDataPaths,
    RealRewardProvider,
    TRAIN_CANDIDATE_ARTIFACT_FILENAME,
    build_real_reward_data_context,
    build_stage5_hybrid_variance_5_15_config,
    create_hybrid_variance_runner,
    resume_hybrid_variance_runner,
)
from factor_gfn.grammar import DAILY_DERIVED_V1_FEATURE_SPACE

FORMAL_MAX_CYCLES = 100
K = 16
config = build_stage5_hybrid_variance_5_15_config(
    max_cycles=FORMAL_MAX_CYCLES,
    trajectories_per_batch=K,
    seed=SEED,
    feature_space=DAILY_DERIVED_V1_FEATURE_SPACE,
)
CONTRACT = {
    'feature_space_id': config.feature_space.feature_space_id,
    'action_count': config.action_registry.action_count,
    'max_cycles': config.training.max_cycles,
    'max_depth': config.search_space.max_depth,
    'max_nodes': config.search_space.max_nodes,
    'K': config.training.trajectories_per_batch,
    'conditions': config.resolved_condition_node_counts,
    'exact_conditions': config.objective.exact_tb_node_counts,
    'lpv_conditions': config.objective.lpv_node_counts,
    'policy_optimizer': config.training.optimizer,
    'policy_lr': config.training.learning_rate,
    'gradient_clip': config.training.model_gradient_clip_norm,
    'seed': config.training.seed,
    'trajectories_per_cycle': config.training.trajectories_per_cycle,
    'optimizer_steps_per_cycle': config.training.optimizer_steps_per_cycle,
    'planned_total_optimizer_steps': config.training.total_optimizer_steps,
    'planned_total_trajectories': config.training.total_training_trajectories,
    'config_fingerprint': config.fingerprint(),
}
assert CONTRACT['feature_space_id'] == 'daily_derived_v1'
assert CONTRACT['action_count'] == 152
assert CONTRACT['max_cycles'] == 100
assert CONTRACT['max_depth'] == 5
assert CONTRACT['max_nodes'] == 15
assert CONTRACT['K'] == 16
assert CONTRACT['conditions'] == tuple(range(1, 16))
assert CONTRACT['exact_conditions'] == (1, 2)
assert CONTRACT['lpv_conditions'] == tuple(range(3, 16))
assert CONTRACT['policy_optimizer'] == 'adam'
assert CONTRACT['policy_lr'] == 1e-4
assert CONTRACT['gradient_clip'] == 5.0
assert CONTRACT['seed'] == 42
assert CONTRACT['trajectories_per_cycle'] == 240
assert CONTRACT['optimizer_steps_per_cycle'] == 15
assert CONTRACT['planned_total_optimizer_steps'] == 1500
assert CONTRACT['planned_total_trajectories'] == 24000
print(json.dumps(CONTRACT, ensure_ascii=False, indent=2), flush=True)

## 3. Step-zero / preflight

只装配真实数据、Derived registry、Trainer、临时 step-0 runner；不调用训练。

In [ ]:
from factor_gfn.grammar import Expression, get_action_id

DATA_CONFIG = RealRewardDataConfig()
DATA_PATHS = RealRewardDataPaths(
    expression_features=ExpressionFeatureSpec.daily_derived(),
)
SOURCE_REGISTRY = (
    PROJECT_ROOT / 'runs' / 'daily_derived_v1' / 'exact_tb_n1_n2'
    / 'exhaustive_registry.sqlite3'
)
RUN_ROOT = (
    PROJECT_ROOT / 'runs' / 'daily_derived_v1'
    / 'stage5_hybrid_variance_real_5_15'
)

required_paths = [
    DATA_PATHS.tensor_path,
    DATA_PATHS.expression_tensor_path,
    DATA_PATHS.expression_metadata_path,
    DATA_PATHS.universe_mask_path,
    DATA_PATHS.date_list_path,
    DATA_PATHS.stock_list_path,
    DATA_PATHS.processed_metadata_path,
    DATA_PATHS.industry_path,
    DATA_PATHS.industry_metadata_path,
    DATA_PATHS.barra_paths.metadata_path,
    DATA_PATHS.barra_paths.market_return_path,
    *[DATA_PATHS.barra_paths.exposure_path(name) for name in (
        'market_beta', 'size', 'momentum', 'volatility', 'liquidity'
    )],
    SOURCE_REGISTRY,
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError({'missing_preflight_paths': missing_paths})

preflight_started = perf_counter()
context = build_real_reward_data_context(DATA_CONFIG, DATA_PATHS)
provider = RealRewardProvider(context, config.reward)
probe_prefix = (
    get_action_id('add', registry=config.action_registry),
    get_action_id('ret_gap', registry=config.action_registry),
    get_action_id('ret_cc1', registry=config.action_registry),
)
probe = Expression.from_prefix(
    probe_prefix,
    action_registry=config.action_registry,
)
probe_assignment = provider.evaluate(probe)
if provider.interpreter_evaluation_count != 1:
    raise RuntimeError('FactorInterpreter preflight 未执行恰好一次 Derived probe')

assert config.search_space.max_depth == 5
assert config.search_space.max_nodes == 15
assert config.resolved_condition_node_counts == tuple(range(1, 16))
assert config.training.trajectories_per_batch == 16
assert config.training.max_cycles == 100
assert config.training.learning_rate == 1e-4
assert config.training.model_gradient_clip_norm == 5.0
assert config.objective.exact_tb_node_counts == (1, 2)
assert config.objective.lpv_node_counts == tuple(range(3, 16))
assert config.training.seed == 42
assert context.expression_feature_space_id == 'daily_derived_v1'
assert tuple(context.ordered_feature_names) == tuple(config.feature_space.ordered_leaf_names)
assert context.manifest['label_formula'] == 'open[t+6] / open[t+1] - 1'
assert provider.manifest()['validation_oos_loaded'] is False

preflight_device = DEVICE if CUDA_READY else 'cpu'
preflight_trainer = HybridVarianceTrainer(config, provider, device=preflight_device)
preflight_registry = ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True, action_registry=config.action_registry)
try:
    exact_semantics = preflight_trainer.target_exhaustive_reuse_semantics()
    exact_proofs = preflight_trainer.configure_hybrid_exhaustive_registry(
        preflight_registry,
        source_semantics_by_N={1: exact_semantics, 2: exact_semantics},
    )
    exact_masses = {
        node_count: preflight_registry.exact_mass_result(node_count)
        for node_count in (1, 2)
    }
    with TemporaryDirectory(prefix='factor_gfn_derived_hybrid_preflight_') as temporary:
        temporary_runner = create_hybrid_variance_runner(
            preflight_trainer,
            Path(temporary) / 'runner',
        )
        temporary_artifact = json.loads(
            temporary_runner.train_candidate_artifact_path.read_text(encoding='utf-8')
        )
        artifact_ready = (
            temporary_artifact['committed_optimizer_step'] == 0
            and temporary_artifact['candidate_count'] == 0
            and temporary_artifact['vocabulary']['feature_space_id'] == 'daily_derived_v1'
            and temporary_runner.latest_checkpoint_path.is_file()
        )
finally:
    preflight_registry.close()

PREFLIGHT = {
    'data_ready': True,
    'feature_space_id': context.expression_feature_space_id,
    'action_count': config.action_registry.action_count,
    'requested_train_range': [DATA_CONFIG.train_start, DATA_CONFIG.train_end],
    'actual_train_range': [context.manifest['actual_train_start'], context.manifest['actual_train_end']],
    'expression_shape': list(context.expression_feature_tensor.shape),
    'evaluation_shape': context.manifest['shape']['evaluation'],
    'label_formula': context.manifest['label_formula'],
    'provider_fingerprint': provider.fingerprint(),
    'context_fingerprint': context.fingerprint,
    'validation_oos_loaded': provider.manifest()['validation_oos_loaded'],
    'factor_interpreter_probe_valid': probe_assignment.valid,
    'factor_interpreter_evaluations': provider.interpreter_evaluation_count,
    'exact_registry_read_only': preflight_registry.read_only,
    'exact_conditions_ready': sorted(exact_proofs),
    'exact_log_z': {n: exact_masses[n].exact_tb_log_z for n in (1, 2)},
    'exact_proof_fingerprints': {n: exact_proofs[n].proof_fingerprint for n in (1, 2)},
    'cuda_ready': CUDA_READY,
    'artifact_writer_ready': artifact_ready,
    'checkpoint_output_root': str(RUN_ROOT),
    'artifact_filename': TRAIN_CANDIDATE_ARTIFACT_FILENAME,
    'optimizer_step_after_preflight': preflight_trainer.optimizer_step,
    'preflight_seconds': perf_counter() - preflight_started,
}
assert PREFLIGHT['action_count'] == 152
assert PREFLIGHT['validation_oos_loaded'] is False
assert PREFLIGHT['exact_conditions_ready'] == [1, 2]
assert PREFLIGHT['artifact_writer_ready']
assert PREFLIGHT['optimizer_step_after_preflight'] == 0
PREFLIGHT_READY = all((
    PREFLIGHT['data_ready'],
    PREFLIGHT['cuda_ready'],
    PREFLIGHT['artifact_writer_ready'],
    PREFLIGHT['exact_conditions_ready'] == [1, 2],
))
print(json.dumps(PREFLIGHT, ensure_ascii=False, indent=2), flush=True)
print({'PREFLIGHT_READY': PREFLIGHT_READY, 'REAL_TRAINING_EXECUTED': False}, flush=True)

## 4. 新建或恢复 Derived run

首次运行保持 `MODE='new'`；恢复时必须显式填写 Derived run目录。

In [ ]:
RUN_REAL_ONE_CYCLE = False
MODE = 'new'
RESUME_RUN_DIR = None

if not RUN_REAL_ONE_CYCLE:
    raise RuntimeError('Safety stop：检查 preflight 后再显式设置 RUN_REAL_ONE_CYCLE=True')
if not PREFLIGHT_READY:
    raise RuntimeError('Derived 正式 one-cycle 因 preflight 未通过而被阻止')
if MODE not in {'new', 'resume'}:
    raise ValueError("MODE 必须是 'new' 或 'resume'")
if MODE == 'new' and RESUME_RUN_DIR is not None:
    raise ValueError('new 模式不得设置 RESUME_RUN_DIR')
if MODE == 'resume' and RESUME_RUN_DIR is None:
    raise ValueError('resume 模式必须显式设置 Derived RESUME_RUN_DIR')

formal_context = build_real_reward_data_context(DATA_CONFIG, DATA_PATHS)
formal_provider = RealRewardProvider(formal_context, config.reward)
formal_trainer = HybridVarianceTrainer(config, formal_provider, device=DEVICE)
exact_registry = ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True, action_registry=config.action_registry)
formal_semantics = formal_trainer.target_exhaustive_reuse_semantics()
formal_trainer.configure_hybrid_exhaustive_registry(
    exact_registry,
    source_semantics_by_N={1: formal_semantics, 2: formal_semantics},
)
if MODE == 'new':
    run_id = 'derived_hybrid_5_15_k16_seed42_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    runner = create_hybrid_variance_runner(formal_trainer, RUN_ROOT / run_id)
else:
    runner = resume_hybrid_variance_runner(Path(RESUME_RUN_DIR), formal_trainer)
print({
    'mode': MODE,
    'run_dir': str(runner.run_dir),
    'optimizer_step': runner.trainer.optimizer_step,
    'cycle_assignment': asdict(runner.trainer.complexity_scheduler.peek()),
    'checkpoint': str(runner.latest_checkpoint_path),
    'artifact': str(runner.train_candidate_artifact_path),
}, flush=True)

## 5. 一轮 smoke

仅允许全新 step-0 run执行一次完整cycle，覆盖N=1..15。

In [ ]:
if not RUN_REAL_ONE_CYCLE:
    raise RuntimeError('Safety stop：one-cycle执行已关闭')
if runner.trainer.optimizer_step != 0:
    raise RuntimeError('Initial one-cycle cell requires a fresh step-0 runner and must not be rerun')
cycle_started = perf_counter()
start_assignment = runner.trainer.complexity_scheduler.peek()
start_cycle = start_assignment.cycle_index
start_optimizer_step = runner.trainer.optimizer_step
batch_runtime = []
if CUDA_READY:
    torch.cuda.reset_peak_memory_stats()
while not runner.complete and runner.trainer.complexity_scheduler.peek().cycle_index == start_cycle:
    batch_started = perf_counter()
    output = runner.run_attempts(1)[0]
    elapsed = perf_counter() - batch_started
    if output.updated:
        batch_runtime.append({
            'condition_N': output.diagnostics.condition_N,
            'objective_kind': output.diagnostics.objective_kind,
            'optimizer_step': output.global_optimizer_step,
            'elapsed_seconds': elapsed,
            'cuda_allocated_bytes': torch.cuda.memory_allocated() if CUDA_READY else None,
            'cuda_peak_bytes': torch.cuda.max_memory_allocated() if CUDA_READY else None,
        })
        print(batch_runtime[-1], flush=True)
cycle_runtime_seconds = perf_counter() - cycle_started
successful_updates = runner.trainer.optimizer_step - start_optimizer_step
assert successful_updates == config.training.optimizer_steps_per_cycle
assert runner.trainer.total_trajectories_seen == config.training.trajectories_per_cycle
assert sorted(item['condition_N'] for item in batch_runtime) == list(range(1, 16))
assert {item['condition_N'] for item in batch_runtime if item['objective_kind'] == 'exact_tb'} == {1, 2}
assert {item['condition_N'] for item in batch_runtime if item['objective_kind'] == 'log_partition_variance'} == set(range(3, 16))
print({'successful_updates': successful_updates, 'cycle_runtime_seconds': cycle_runtime_seconds, 'run_dir': str(runner.run_dir)}, flush=True)

## 6. 一轮诊断

检查Exact-TB、LPV与公共训练诊断字段，不据此调整冻结参数。

In [ ]:
diagnostics_by_N = {item.condition_N: item.to_dict() for item in runner.trainer.diagnostic_history[-15:]}
exact_fields = ('exact_log_z', 'tb_loss', 'tb_delta_mean', 'tb_delta_std', 'tb_delta_rms')
lpv_fields = ('zeta_mean', 'zeta_std', 'zeta_variance', 'variance_loss', 'centered_zeta_rms', 'unique_terminal_count', 'unique_terminal_fraction')
common_fields = ('reward_mean', 'sum_log_pf_mean', 'sum_log_pb_mean', 'policy_grad_norm', 'requested_count', 'accepted_count', 'invalid_count', 'retry_count', 'retry_exhausted_count', 'trajectories_in_batch', 'global_optimizer_step', 'condition_position_in_cycle')
for condition_N in sorted(diagnostics_by_N):
    row = diagnostics_by_N[condition_N]
    objective_fields = exact_fields if condition_N in (1, 2) else lpv_fields
    print({'condition_N': condition_N, **{name: row[name] for name in (*objective_fields, *common_fields)}}, flush=True)
print({'per_N_runtime': batch_runtime, 'full_cycle_runtime_seconds': cycle_runtime_seconds}, flush=True)

## 7. Derived candidate artifact

检查候选记录、提交step和Derived vocabulary identity。

In [ ]:
artifact = json.loads(runner.train_candidate_artifact_path.read_text(encoding='utf-8'))
assert artifact['committed_optimizer_step'] == runner.trainer.optimizer_step
assert artifact['candidate_count'] == len(artifact['records'])
assert artifact['vocabulary']['feature_space_id'] == 'daily_derived_v1'
assert artifact['vocabulary']['action_space_fingerprint'] == config.action_registry.fingerprint()
assert all(record['vocabulary'] == artifact['vocabulary'] for record in artifact['records'])
sample_fields = ('structural_hash', 'formula', 'prefix_token_ids', 'train_ic', 'train_direction', 'train_long_ir', 'train_long_excess_dates', 'train_long_excess_values', 'train_barra_ts_corr', 'first_seen', 'last_seen', 'visit_count')
sample = [{name: record[name] for name in sample_fields} for record in artifact['records'][:3]]
print({'artifact_path': str(runner.train_candidate_artifact_path), 'committed_optimizer_step': artifact['committed_optimizer_step'], 'candidate_count': artifact['candidate_count'], 'vocabulary': artifact['vocabulary'], 'sample': sample}, flush=True)

## 8. Resume 等价验证

使用相同Derived数据、config和只读exact registry重建Trainer并恢复checkpoint。

In [ ]:
saved = {
    'model': {name: value.detach().cpu().clone() for name, value in runner.trainer.model.state_dict().items()},
    'optimizer_step': runner.trainer.optimizer_step,
    'total_trajectories_seen': runner.trainer.total_trajectories_seen,
    'scheduler': runner.trainer.complexity_scheduler.state_dict(),
    'artifact_step': artifact['committed_optimizer_step'],
}
resume_context = build_real_reward_data_context(DATA_CONFIG, DATA_PATHS)
resume_provider = RealRewardProvider(resume_context, config.reward)
resume_trainer = HybridVarianceTrainer(config, resume_provider, device=DEVICE)
resume_registry = ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True, action_registry=config.action_registry)
resume_semantics = resume_trainer.target_exhaustive_reuse_semantics()
resume_trainer.configure_hybrid_exhaustive_registry(
    resume_registry,
    source_semantics_by_N={1: resume_semantics, 2: resume_semantics},
)
resumed_runner = resume_hybrid_variance_runner(runner.run_dir, resume_trainer)
assert resumed_runner.trainer.optimizer_step == saved['optimizer_step']
assert resumed_runner.trainer.total_trajectories_seen == saved['total_trajectories_seen']
assert resumed_runner.trainer.complexity_scheduler.state_dict() == saved['scheduler']
assert all(torch.equal(value.detach().cpu(), saved['model'][name]) for name, value in resumed_runner.trainer.model.state_dict().items())
resumed_artifact = json.loads(resumed_runner.train_candidate_artifact_path.read_text(encoding='utf-8'))
assert resumed_artifact['committed_optimizer_step'] == saved['artifact_step']
assert resumed_artifact['vocabulary'] == artifact['vocabulary']
print({'resume_verified': True, 'optimizer_step': resumed_runner.trainer.optimizer_step, 'trajectories': resumed_runner.trainer.total_trajectories_seen, 'artifact_step': resumed_artifact['committed_optimizer_step']}, flush=True)

## 9. 累计到100 cycles

只有在one-cycle与resume验收通过并得到人工批准后，才启用累计目标训练。

In [ ]:
TARGET_CYCLE = 100
RUN_TO_TARGET_CYCLE = False

def run_until_cycle(active_runner, target_cycle):
    if isinstance(target_cycle, bool) or not isinstance(target_cycle, int) or target_cycle < 1:
        raise ValueError('target_cycle必须是正整数')
    training = active_runner.trainer.config.training
    if target_cycle > training.max_cycles:
        raise ValueError(f'target_cycle={target_cycle}超过冻结预算{training.max_cycles}')
    steps_per_cycle = training.optimizer_steps_per_cycle
    start_optimizer_step = active_runner.trainer.optimizer_step
    start_cycle = start_optimizer_step // steps_per_cycle
    start_position_in_cycle = start_optimizer_step % steps_per_cycle
    target_optimizer_step = target_cycle * steps_per_cycle
    if target_optimizer_step < start_optimizer_step:
        raise ValueError(f'target_cycle={target_cycle}落后于当前optimizer step {start_optimizer_step}')
    attempted_updates = 0
    successful_updates = 0
    continuation_started = perf_counter()
    while active_runner.trainer.optimizer_step < target_optimizer_step:
        if active_runner.complete:
            raise RuntimeError('runner在达到target_cycle前提前complete')
        update_started = perf_counter()
        output = active_runner.run_attempts(1)[0]
        attempted_updates += 1
        if output.updated:
            successful_updates += 1
            print({
                'cycle_index': output.diagnostics.cycle_index,
                'condition_N': output.diagnostics.condition_N,
                'optimizer_step': output.global_optimizer_step,
                'elapsed_seconds': perf_counter() - update_started,
                'objective_kind': output.diagnostics.objective_kind,
                'policy_grad_norm_pre_clip': output.diagnostics.policy_grad_norm,
            }, flush=True)
    expected_updates = target_optimizer_step - start_optimizer_step
    if active_runner.trainer.optimizer_step - start_optimizer_step != expected_updates:
        raise RuntimeError('optimizer-step推进量与累计目标不一致')
    if active_runner.trainer.total_trajectories_seen != target_cycle * training.trajectories_per_cycle:
        raise RuntimeError('trajectory计数与累计target_cycle不一致')
    final_artifact = json.loads(active_runner.train_candidate_artifact_path.read_text(encoding='utf-8'))
    if final_artifact['committed_optimizer_step'] != active_runner.trainer.optimizer_step:
        raise RuntimeError('artifact committed step differs from the runner optimizer step')
    summary = {
        'start_cycle': start_cycle,
        'start_position_in_cycle': start_position_in_cycle,
        'target_cycle': target_cycle,
        'equivalent_cycles_completed_this_call': expected_updates / steps_per_cycle,
        'attempts': attempted_updates,
        'successful_optimizer_updates': successful_updates,
        'final_optimizer_step': active_runner.trainer.optimizer_step,
        'final_trajectories': active_runner.trainer.total_trajectories_seen,
        'final_candidate_count': final_artifact['candidate_count'],
        'runner_complete': active_runner.complete,
        'continuation_seconds': perf_counter() - continuation_started,
        'run_dir': str(active_runner.run_dir),
    }
    print(summary, flush=True)
    return summary

if RUN_TO_TARGET_CYCLE:
    active_runner = resumed_runner if 'resumed_runner' in globals() else runner
    continuation_summary = run_until_cycle(active_runner, target_cycle=TARGET_CYCLE)
    runner = active_runner
else:
    print({'continuation_enabled': False, 'target_cycle': TARGET_CYCLE}, flush=True)